load verified coordinates and the source tables for the real-data causal analysis

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
from collections import Counter
import numpy as np
from IPython.display import Markdown, display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from reasyn.fibril_sources import validate_sources, write_csv, write_json, canonical_bytes
from reasyn.fibril_sampling import stratified_causal_weights, effective_sample_size, hierarchical_draw

manifest, curated, curation = validate_sources(root)
sources = {'article.html': {'path': 'data/raw/publications/frey_2026/article.html', 'url': 'https://elifesciences.org/reviewed-preprints/112692', 'sha256': '32ed63159c77e21ee19ca1b9aa3213ccf0218eb59539560b132a8e68ef0e18ea'}, 'table1.jpg': {'path': 'data/raw/publications/frey_2026/table1.jpg', 'url': 'https://iiif.elifesciences.org/epp/112692%2Fv1%2Fcontent%2F737774v1_tbl1.tif/full/max/0/default.jpg', 'sha256': '27ae50c3f8c83e6b5384e7f9809cb683d3ceb7a9bece3113d90890de6174ac5c'}, 'table1_continued.jpg': {'path': 'data/raw/publications/frey_2026/table1_continued.jpg', 'url': 'https://iiif.elifesciences.org/epp/112692%2Fv1%2Fcontent%2F737774v1_tbl1a.tif/full/max/0/default.jpg', 'sha256': '429e462a7bb7beb26085f7f9b6897b13f7632193b299de646faf7b8888f48657'}, 'table_s1.jpg': {'path': 'data/raw/publications/frey_2026/table_s1.jpg', 'url': 'https://iiif.elifesciences.org/epp/112692%2Fv1%2Fcontent%2F737774v1_tbls1.tif/full/max/0/default.jpg', 'sha256': '6691eb561fb3c9d45247b91e665722d85305d8a9e7a17d06d30dc1e5a9ff1f6c'}}
for source in sources.values():
    assert hashlib.sha256((root / source["path"]).read_bytes()).hexdigest() == source["sha256"]
study_doi = "10.7554/elife.112692.1"
study = [row for row in curated if row["study_doi"].lower() == study_doi]
assert len(study) == 23

transcribe sample-level conditions and map deposits through the published emdb identifiers

In [ ]:
sample_values = {
    12: (5.8, 7, "Era 1", "SP", None),
    19: (6.3, 9, "Era 1", "SP", None),
    23: (7.0, 9, "Era 1", "HPLC", 0),
    44: (7.4, 12, "Era 1", "HPLC", 0),
    45: (7.0, 13, "Era 1", "HPLC", 0),
    46: (7.0, 13, "Era 1", "HPLC", 5),
    48: (7.0, 13, "Era 1", "HPLC", 50),
    49: (7.0, 13, "Era 1", "HPLC", 100),
    56: (7.0, 15, "Era 2", "HPLC", 50),
    58: (7.0, 15, "Era 2", "HPLC", 100),
    59: (7.4, 16, "Era 2", "HPLC", 0),
    60: (7.4, 16, "Era 2", "HPLC", 5),
    62: (7.4, 16, "Era 2", "HPLC", 100),
}
emdb_samples = {
    "58905": 12, "58913": 19, "58821": 23, "58830": 23,
    "58842": 23, "58845": 23, "58924": 44, "58925": 44,
    "58914": 45, "58917": 45, "58918": 45, "58919": 45,
    "58921": 46, "58922": 48, "58923": 49, "58911": 56,
    "58912": 58, "58906": 59, "58907": 59, "58908": 60,
    "58926": 60, "58909": 62,
}
linked, unresolved = [], []
for row in study:
    entry_path = f"data/raw/alpha_synuclein_fibrils/pdb_metadata/{row['pdb_id']}.json"
    entry = json.loads((root / entry_path).read_text())
    matches = {emdb_samples[identifier.removeprefix("EMD-")]
               for identifier in entry["rcsb_entry_container_identifiers"]["emdb_ids"]
               if identifier.removeprefix("EMD-") in emdb_samples}
    if len(matches) != 1:
        unresolved.append({"pdb_id": row["pdb_id"], "reason": "no unique Table S1 EMDB-to-sample mapping"})
        continue
    sample = matches.pop()
    ph, batch, era, purity, truncation = sample_values[sample]
    linked.append({
        "pdb_id": row["pdb_id"], "study_id": study_doi,
        "sample_id": f"frey_2026_sample_{sample}", "sample_number": sample,
        "batch_id": f"frey_2026_batch_{batch}", "growth_ph": ph,
        "preparation_era": era, "purity_category": purity,
        "added_truncation_percent": truncation if truncation is not None else "unknown",
        "coordinate_path": row["coordinate_path"],
        "coordinate_sha256": manifest["files"][row["coordinate_path"]]["sha256"],
        "sample_mapping_source": sources["table_s1.jpg"]["path"] + "::EMDB entry",
        "condition_source": sources["table1.jpg" if sample <= 40 else "table1_continued.jpg"]["path"] + f"::sample {sample}",
        "deposit_source": entry_path + "::rcsb_entry_container_identifiers.emdb_ids",
        "review_status": "assistant transcription; not human approved",
    })
assert len(linked) == 22
assert {row["pdb_id"] for row in unresolved} == {"33XC"}
assert next(row for row in linked if row["pdb_id"] == "32IT")["sample_number"] == 60
assert next(row for row in linked if row["pdb_id"] == "32IS")["sample_number"] == 44
mapping_notes = {
    "32IT": "Table S1 lists 32IS for EMD-58926; downloaded PDB metadata identifies 32IT",
    "32IS": "downloaded PDB metadata links 32IS to EMD-58925 and sample 44",
    "33XC": "Table S1 sample 15 names 32HW / EMD-58903; current 33XC links EMD-59649; sample identity remains unresolved",
}
sample_rows = []
for sample_id in sorted({row["sample_id"] for row in linked}):
    members = [row for row in linked if row["sample_id"] == sample_id]
    first = members[0]
    sample_rows.append({key: first[key] for key in ("study_id", "sample_id", "batch_id", "growth_ph",
        "preparation_era", "purity_category", "added_truncation_percent", "condition_source", "review_status")}
        | {"pdb_ids": json.dumps([row["pdb_id"] for row in members]), "conformation_count": len(members)})
len(sample_rows), unresolved